# Dashboard v2 interaction repair v1

Purpose: verify the local dashboard v2 path after repairing rerun window registration, auto-update task visibility, buy/sell feature comparison ordering, and heavy hover feature snapshots.

Scope: local `scripts/dashboard_server.py` workflow only. Vercel/remote deploy is intentionally out of scope.

In [1]:
from pathlib import Path
import json

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'dashboard_data').exists():
            return candidate
    raise FileNotFoundError('could not locate repo root')

ROOT = find_repo_root(Path.cwd())
MANIFEST = ROOT / 'dashboard_data' / 'v2' / 'manifest.json'
MODEL_DIR = ROOT / 'dashboard_data' / 'v2' / 'models'
SCREENSHOT = ROOT / 'runtime' / 'harness' / 'card' / 'dashboard_v2_interaction_repair_v1' / 'dashboard_v2_bundle_verified.png'

manifest = json.loads(MANIFEST.read_text())
models = {profile: json.loads((MODEL_DIR / f'{profile}.json').read_text()) for profile in manifest['available_profiles']}
summary = {
    'available_profiles': manifest['available_profiles'],
    'default_comparison_profiles': manifest['default_comparison_profiles'],
    'default_chart_profile': manifest.get('default_chart_profile'),
    'safe_backtest_end': manifest.get('safe_backtest_end'),
    'auto_update_configured': manifest.get('auto_update', {}).get('configured'),
    'auto_update_tasks': [task['name'] for task in manifest.get('auto_update', {}).get('tasks', [])],
    'replay_ready': manifest.get('auto_update', {}).get('model_replay_freshness', {}).get('ready_count'),
    'replay_profile_count': manifest.get('auto_update', {}).get('model_replay_freshness', {}).get('profile_count'),
    'runnable_profiles': manifest.get('profile_groups', {}).get('runnable', []),
    'reference_profiles': manifest.get('profile_groups', {}).get('reference', []),
    'evidence_only_profiles': manifest.get('profile_groups', {}).get('evidence_only', []),
    'screenshot_exists': SCREENSHOT.exists(),
    'screenshot_path': str(SCREENSHOT.relative_to(ROOT)),
}
summary

{'available_profiles': ['soxl_backtest_policy_reimplementation_v1',
  'balanced_rule_residual_v19_moe_v2'],
 'default_comparison_profiles': ['soxl_backtest_policy_reimplementation_v1',
  'balanced_rule_residual_v19_moe_v2'],
 'default_chart_profile': 'balanced_rule_residual_v19_moe_v2',
 'safe_backtest_end': '2026-06-09',
 'auto_update_configured': True,
 'auto_update_tasks': ['preopen', 'postclose', 'dashboard_refresh'],
 'replay_ready': 0,
 'replay_profile_count': 7,
 'runnable_profiles': [],
 'reference_profiles': ['soxl_backtest_policy_reimplementation_v1'],
 'evidence_only_profiles': ['balanced_rule_residual_v19_moe_v2'],
 'screenshot_exists': True,
 'screenshot_path': 'runtime/harness/card/dashboard_v2_interaction_repair_v1/dashboard_v2_bundle_verified.png'}

In [2]:
assert summary['auto_update_configured'] is True
assert 'dashboard_refresh' in summary['auto_update_tasks']
assert summary['replay_ready'] == 0
assert summary['replay_profile_count'] == 7
assert summary['screenshot_exists'] is True

posttrain_hits = []
for path in (ROOT / 'dashboard_data').rglob('*.json'):
    text = path.read_text(errors='ignore')
    if 'post_train_review' in text or 'PostTrain' in text or 'posttrain' in text.lower():
        posttrain_hits.append(str(path.relative_to(ROOT)))
assert not posttrain_hits, posttrain_hits

schema_readout = {
    profile: {
        'source_kind': payload['source_kind'],
        'runnable': payload['runnable'],
        'runnable_reason': payload.get('runnable_reason'),
        'feature_count': len(payload.get('feature_columns', [])),
        'snapshot_count': len(payload.get('snapshot_columns', [])),
        'top_buy_sell_features': [row['feature'] for row in payload.get('feature_action_summary', [])[:4]],
    }
    for profile, payload in models.items()
}
assert len({tuple(item['top_buy_sell_features']) for item in schema_readout.values()}) > 1
schema_readout

{'soxl_backtest_policy_reimplementation_v1': {'source_kind': 'soxlbacktest_qlib_buysell',
  'runnable': False,
  'runnable_reason': 'reference_baseline',
  'feature_count': 11,
  'snapshot_count': 11,
  'top_buy_sell_features': ['panic_score', 'close', 'open', 'high']},
 'balanced_rule_residual_v19_moe_v2': {'source_kind': 'retained_dashboard_export',
  'runnable': False,
  'runnable_reason': 'evidence_only_retained_dashboard_export',
  'feature_count': 59,
  'snapshot_count': 59,
  'top_buy_sell_features': ['relative_volume_zscore',
   'sharpe_proxy_21',
   'relative_strength_21',
   'drawdown_streak_63']}}

## Conclusion

- The verified local entrypoint is `scripts/dashboard_server.py`, not the ordinary static `python -m http.server` process that was still serving the old dashboard on another port.
- The v2 dashboard bundle renders from `/dashboard.html -> /web/dashboard-v2/dist/index.html` with K-line candles, SOXL price, position, NAV, operations status, and buy/sell feature summary visible in the screenshot sidecar.
- Auto-update tasks are configured and visible, but replayable retained actors are still `0 / 7`; selected models are therefore correctly shown as reference/evidence-only rather than runnable inference models.
- Model feature schemas differ by profile and the buy/sell feature comparison now surfaces top rows per model before schema details.
- No `post_train_review` / PostTrainPreview payload remains in dashboard JSON data.

Next action: build a retained walk-forward-passing actor before enabling true interactive inference for a model; do not label reference baselines or retained static exports as runnable.